<div style="border-left:6px solid #2b6cb0;padding:0.8em 1.2em;background:#f0f6ff">
<b>ECAC project · Section 10.2</b><br>
<span style="font-size:1.4em"><b>Pre-trained feature extractors</b></span><br><br>
Three frozen models turn each song into a row of numbers: PANNs CNN14 for the audio (2048), all-MiniLM-L6-v2 for the lyrics (384) and ResNet-50 for the cover (2048). This notebook checks the installation, passes one song through each model, extracts the whole dataset and loads the result. The README describes what each extractor does to its input.
</div>

## 1. Setup

Install the requirements once, in a terminal, from the folder that holds `requirements.txt`:

```bash
pip install -r requirements.txt
```

The cell below makes the package importable from this notebook.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # the folder that holds ecac_extractors/
from ecac_extractors.cli import main

## 2. Check the installation

The first run downloads the weights (about 520 MB in all). Each model is then tested on a known input: the last line should be `everything works`.

In [ ]:
main(["check"])

## 3. One song through each model

Set `MEDIA` to the folder that holds `audio/`, `lyrics/` and `album_covers/`.

In [ ]:
MEDIA = Path("C:/Users/laran/ECAC-Projeto/dataset/dataset")            # change this

audio_file = sorted((MEDIA / "audio").rglob("*.mp3"))[0]
lyrics_file = sorted((MEDIA / "lyrics").rglob("*.txt"))[0]
cover_file = sorted((MEDIA / "album_covers").rglob("*.jpg"))[0]
print(audio_file.name, lyrics_file.name, cover_file.name)

In [ ]:
from ecac_extractors import AudioEmbedder, CoverEmbedder, LyricsEmbedder

audio, lyrics, covers = AudioEmbedder(), LyricsEmbedder(), CoverEmbedder()   # load once, reuse

In [ ]:
r = audio.embed_file(audio_file)
print("audio:", r["mean"].shape, r["info"])
print("what CNN14 hears:", audio.top_tags(audio_file))

In [ ]:
r = lyrics.embed_file(lyrics_file)
print("lyrics:", r["vector"].shape, r["info"])

In [ ]:
from PIL import Image

display(Image.open(cover_file).resize((200, 200)))
print("cover:", covers(cover_file).shape)
print("what ResNet-50 sees:", covers.top_classes(cover_file))

The classes printed above are what each network was trained to recognise. Comparing them with the quadrant of the song is a first look at Section 10.3: how far is each training task from emotion?

## 4. The whole dataset

One CSV per domain in `features/`, and `models_used.json` with the exact versions and parameters. The audio takes a few tens of minutes on a CPU. If the run stops, run the same cell again: files already written are skipped.

In [ ]:
main(["extract", "--media", str(MEDIA), "--out", "features"])

## 5. The result

In [ ]:
import json
import pandas as pd

A = pd.read_csv("features/audio_panns.csv")
L = pd.read_csv("features/lyrics_sbert.csv")
C = pd.read_csv("features/covers_resnet50.csv")
for name, d in (("audio", A), ("lyrics", L), ("covers", C)):
    print(f"{name:7s} {d.shape[0]:5d} files x {d.shape[1] - 1} numbers   id column: {d.columns[0]}")

used = json.load(open("features/models_used.json"))
for domain in ("audio", "lyrics", "covers"):
    print(domain, "failed:", [f["file"] for f in used[domain]["failed"]])

The rows are keyed by file name; joining them with `metadata.csv`, and deciding what to do with duplicated recordings and missing covers, are part of Sections 2.2, 4.1 and 4.5.

<div style="border-left:6px solid #b8410f;padding:0.6em 1em;background:#fdeee6">
<b>Protocol (Section 10.6).</b> These models fit nothing on the songs, so their vectors can be computed once for every song, test songs included. Everything fitted after them (the scaling, the weights of the blocks, PCA, LDA, the selection) is fitted inside the training folds only.
</div>